In [1]:
!pip install brian2 --quiet
from brian2 import *
import matplotlib.pyplot as plt

# Set global parameters
tau = 10*ms
V_reset = 0

# =============================================================================
# 1. AND Gate Implementation
# =============================================================================
def simulate_AND():
    print("\nAND Gate Results:")
    V_th = 1.5  # Threshold
    w = 1.0      # Synaptic weights

    for A, B in [(0,0), (0,1), (1,0), (1,1)]:
        start_scope()
        eqs = '''
        dv/dt = (I - v)/tau : 1
        I : 1
        '''
        neuron = NeuronGroup(1, eqs, threshold='v > V_th', reset='v = V_reset', method='exact')
        neuron.I = w*A + w*B
        spike_mon = SpikeMonitor(neuron)
        run(20*ms)
        print(f"Input ({A},{B}) → Output: {1 if spike_mon.count[0] > 0 else 0}")

# =============================================================================
# 2. OR Gate Implementation
# =============================================================================
def simulate_OR():
    print("\nOR Gate Results:")
    V_th = 0.8  # Lower threshold
    w = 1.0

    for A, B in [(0,0), (0,1), (1,0), (1,1)]:
        start_scope()
        eqs = '''
        dv/dt = (I - v)/tau : 1
        I : 1
        '''
        neuron = NeuronGroup(1, eqs, threshold='v > V_th', reset='v = V_reset', method='exact')
        neuron.I = w*A + w*B
        spike_mon = SpikeMonitor(neuron)
        run(20*ms)
        print(f"Input ({A},{B}) → Output: {1 if spike_mon.count[0] > 0 else 0}")

# =============================================================================
# 3. NOT Gate Implementation
# =============================================================================
def simulate_NOT():
    print("\nNOT Gate Results:")
    V_th = 0.5
    w = -1.0  # Inhibitory weight
    bias = 1.0

    for A in [0, 1]:
        start_scope()
        eqs = '''
        dv/dt = (I - v)/tau : 1
        I : 1
        '''
        neuron = NeuronGroup(1, eqs, threshold='v > V_th', reset='v = V_reset', method='exact')
        neuron.I = w*A + bias
        spike_mon = SpikeMonitor(neuron)
        run(20*ms)
        print(f"Input {A} → Output: {1 if spike_mon.count[0] > 0 else 0}")

# =============================================================================
# 4. XOR Gate Implementation (Two-Layer Network)
# =============================================================================
def simulate_XOR():
    print("\nXOR Gate Results:")

    # First layer parameters
    V_th_layer1 = 1.0
    weights_layer1 = [(1.5, -1.0), (-1.0, 1.5)]  # (A∧¬B) and (¬A∧B)

    # Second layer (OR gate)
    V_th_layer2 = 0.9

    for A, B in [(0,0), (0,1), (1,0), (1,1)]:
        start_scope()

        # Layer 1
        layer1_eqs = '''
        dv/dt = (I - v)/tau : 1
        I : 1
        '''
        layer1 = NeuronGroup(2, layer1_eqs, threshold='v > V_th_layer1', reset='v = V_reset', method='exact')
        layer1.I = [weights_layer1[0][0]*A + weights_layer1[0][1]*B,
                    weights_layer1[1][0]*A + weights_layer1[1][1]*B]

        # Layer 2 (OR)
        layer2 = NeuronGroup(1, layer1_eqs, threshold='v > V_th_layer2', reset='v = V_reset', method='exact')
        syn = Synapses(layer1, layer2, 'w : 1', on_pre='v_post += w')
        syn.connect(i=0, j=0); syn.w[0] = 1.0
        syn.connect(i=1, j=0); syn.w[1] = 1.0

        spike_mon = SpikeMonitor(layer2)
        run(20*ms)
        print(f"Input ({A},{B}) → Output: {1 if spike_mon.count[0] > 0 else 0}")

# =============================================================================
# Run All Simulations
# =============================================================================
simulate_AND()
simulate_OR()
simulate_NOT()
simulate_XOR()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.2 MB/s eta 0:00:00

AND Gate Results:
Input (0,0) → Output: 0
Input (0,1) → Output: 0
Input (1,0) → Output: 0
Input (1,1) → Output: 1

OR Gate Results:
Input (0,0) → Output: 0
Input (0,1) → Output: 1
Input (1,0) → Output: 1
Input (1,1) → Output: 1

NOT Gate Results:
Input 0 → Output: 1
Input 1 → Output: 0

XOR Gate Results:
Input (0,0) → Output: 0
Input (0,1) → Output: 1
Input (1,0) → Output: 1
Input (1,1) → Output: 0
